## DenseNet Architecture

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------
# Dense Layer (Bottleneck Layer)
# -------------------------------
class _DenseLayer(nn.Module):
    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate):
        """
        Args:
            num_input_features: Number of input feature maps.
            growth_rate: Number of feature maps to add per dense layer.
            bn_size: Multiplicative factor for the bottleneck layer.
            drop_rate: Dropout rate after convolutions.
        """
        super(_DenseLayer, self).__init__()
        self.norm1 = nn.BatchNorm2d(num_input_features)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(num_input_features, bn_size * growth_rate,
                               kernel_size=1, stride=1, bias=False)

        self.norm2 = nn.BatchNorm2d(bn_size * growth_rate)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(bn_size * growth_rate, growth_rate,
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.drop_rate = drop_rate

    def forward(self, x):
        # Bottleneck: BN -> ReLU -> 1x1 Conv
        out = self.conv1(self.relu1(self.norm1(x)))
        # 3x3 Conv: BN -> ReLU -> 3x3 Conv
        out = self.conv2(self.relu2(self.norm2(out)))
        if self.drop_rate > 0:
            out = F.dropout(out, p=self.drop_rate, training=self.training)
        # Concatenate input and output (dense connectivity)
        out = torch.cat([x, out], 1)
        return out

# -------------------------------
# Dense Block
# -------------------------------
class _DenseBlock(nn.Module):
    def __init__(self, num_layers, num_input_features, growth_rate, bn_size, drop_rate):
        """
        Args:
            num_layers: Number of dense layers in the block.
            num_input_features: Number of input feature maps.
            growth_rate: Growth rate of the network.
            bn_size: Bottleneck size.
            drop_rate: Dropout rate.
        """
        super(_DenseBlock, self).__init__()
        layers = []
        for i in range(num_layers):
            layer = _DenseLayer(
                num_input_features + i * growth_rate,
                growth_rate=growth_rate,
                bn_size=bn_size,
                drop_rate=drop_rate,
            )
            layers.append(layer)
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

# -------------------------------
# Transition Layer
# -------------------------------
class _Transition(nn.Module):
    def __init__(self, num_input_features, num_output_features):
        """
        Args:
            num_input_features: Number of input feature maps.
            num_output_features: Number of output feature maps (usually reduced).
        """
        super(_Transition, self).__init__()
        self.norm = nn.BatchNorm2d(num_input_features)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(num_input_features, num_output_features,
                              kernel_size=1, stride=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        out = self.conv(self.relu(self.norm(x)))
        out = self.pool(out)
        return out

# -------------------------------
# DenseNet Model
# -------------------------------
class DenseNet(nn.Module):
    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16),
                 num_init_features=64, bn_size=4, drop_rate=0, num_classes=1000):
        """
        Args:
            growth_rate: How many filters to add each layer (k in the paper).
            block_config: Number of layers in each dense block.
                          For DenseNet-121, block_config=(6, 12, 24, 16).
            num_init_features: Number of filters in the initial convolution layer.
            bn_size: Multiplicative factor for bottleneck layers.
            drop_rate: Dropout rate.
            num_classes: Number of classes for classification.
        """
        super(DenseNet, self).__init__()

        # Initial convolution and pooling
        self.features = nn.Sequential(
            nn.Conv2d(3, num_init_features, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(num_init_features),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Build Dense Blocks with Transition Layers
        num_features = num_init_features
        self.dense_blocks = nn.ModuleList()
        self.transitions = nn.ModuleList()
        for i, num_layers in enumerate(block_config):
            block = _DenseBlock(num_layers, num_features, growth_rate, bn_size, drop_rate)
            self.dense_blocks.append(block)
            num_features = num_features + num_layers * growth_rate
            # Add transition layer between blocks (except after the last block)
            if i != len(block_config) - 1:
                trans = _Transition(num_features, num_features // 2)
                self.transitions.append(trans)
                num_features = num_features // 2

        # Final batch norm
        self.norm_final = nn.BatchNorm2d(num_features)
        # Classifier head
        self.classifier = nn.Linear(num_features, num_classes)

        # Weight initialization (optional)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Initial feature extraction
        out = self.features(x)
        # Dense blocks with transitions
        for i, block in enumerate(self.dense_blocks):
            out = block(out)
            if i < len(self.transitions):
                out = self.transitions[i](out)
        # Final batch norm and ReLU
        out = self.norm_final(out)
        out = F.relu(out, inplace=True)
        # Global average pooling over the feature map
        out = F.adaptive_avg_pool2d(out, (1, 1)).view(out.size(0), -1)
        out = self.classifier(out)
        return out

# -------------------------------
# Testing the DenseNet Model
# -------------------------------
if __name__ == '__main__':
    # Example: Create a DenseNet-121 like model (using block_config=(6, 12, 24, 16))
    model = DenseNet(growth_rate=32, block_config=(6, 12, 24, 16),
                     num_init_features=64, bn_size=4, drop_rate=0, num_classes=10)
    # Dummy input: Batch of 2 images with 3 channels and 224x224 resolution
    x = torch.randn(2, 3, 224, 224)
    y = model(x)
    print("Output shape:", y.shape)  # Expected output shape: (2, 10)


Output shape: torch.Size([2, 10])
